In [ ]:
# !pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# 1. 실제 이커머스 데이터(Online Retail) 로드
# 영국 온라인 쇼핑몰의 1년치 실제 결제 데이터입니다.
df = pd.read_excel('Online Retail.xlsx')

# 2. 데이터 전처리 (플라이휠 지표 추출)
# 결측치 제거 및 취소 주문 제외
df.dropna(subset=['CustomerID'], inplace=True)
df = df[df['Quantity'] > 0]
df['TotalSum'] = df['Quantity'] * df['UnitPrice']

# [핵심] RFM 지표 생성 (플라이휠의 엔진 상태 확인)
# Recency(최근성), Frequency(빈도), Monetary(구매금액)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
customers = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # 최근 구매일 (Retention)
    'InvoiceNo': 'count',                                   # 구매 빈도 (Engagement)
    'TotalSum': 'sum'                                       # 총 구매 금액 (Revenue)
})

customers.rename(columns={
    'InvoiceDate': 'Recency(최근성)',
    'InvoiceNo': 'Frequency(빈도)',
    'TotalSum': 'Monetary(총지출)'
}, inplace=True)

# 3. 타겟 변수 설정 (미래 가치 예측)
# 여기서는 '총지출' 자체를 예측하거나, 로그 변환을 통해 미래 가치를 예측하는 모델로 가정합니다.
X = customers[['Recency(최근성)', 'Frequency(빈도)']]
y = customers['Monetary(총지출)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. 랜덤 포레스트(ML) 모델 학습
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

# 5. 피처 임포턴스 (Feature Importance) - "플라이휠의 어떤 축이 매출에 가장 기여하는가?"
importances = pd.DataFrame({
    '지표': X.columns,
    '기여도': model_rf.feature_importances_
}).sort_values(by='기여도', ascending=False)

print("\n🎡 [플라이휠 분석] 고객 가치 예측 핵심 요인")
print(importances)

# 6. 예측 결과 샘플
y_pred = model_rf.predict(X_test)
result_sample = pd.DataFrame({'실제매출': y_test, '예측매출': y_pred}).head(5)
print("\n💎 [예측 결과] 이 고객은 우리 서비스에 얼마나 기여할 것인가?")
print(result_sample.round(0))


실제 이커머스 데이터를 불러오는 중입니다... (약 10~20초 소요)

🎡 [플라이휠 분석] 고객 가치 예측 핵심 요인
              지표       기여도
1  Frequency(빈도)  0.816349
0   Recency(최근성)  0.183651

💎 [예측 결과] 이 고객은 우리 서비스에 얼마나 기여할 것인가?
              실제매출    예측매출
CustomerID                
17785.0      132.0   440.0
14317.0      509.0   374.0
15977.0     1055.0  1003.0
12364.0     1313.0  1975.0
14565.0     3099.0  1550.0
